In [1]:
import jax
from jax import numpy as np, random as jr, Array
import zodiax as zdx
import dLux as dl
import dLuxToliman as dlT

# Enable 64bit precision (note this must be run in the first cell of the notebook)
jax.config.update("jax_enable_x64", True)


# plotting
import matplotlib as mpl
from matplotlib import pyplot as plt

# plt.style.use(["science", "no-latex"])
plt.rcParams["image.origin"] = "lower"
plt.rcParams["figure.dpi"] = 150

In [2]:
oversample = 6
det_npixels = 128
radial_orders = [2, 3]
det_pscale = 0.375

norm_params = {
    "r": 2,
    # "r": 0.25e-4,
    "shear": 0.25,
    "phi": 0.0,
    "kernel_size": 17,
}

# # Creating common optical system
# optics = dlT.TolimanOpticalSystem(
#     oversample=oversample,
#     psf_npixels=det_npixels,
#     radial_orders=radial_orders,
#     psf_pixel_scale=det_pscale,
# )
# optics = optics.divide("aperture.basis", 1e9)  # Set basis units to nanometers

# # Creating common source
# src = dlT.sources.AlphaCen(
#     separation=np.array(10.0),
#     position_angle=np.array(90.0),
#     x_position=np.array(0.0),
#     y_position=np.array(0.0),
#     log_flux=np.array(7.581),
#     contrast=np.array(3.37),
# )

norm_det = dl.LayeredDetector(
    [
        ("Jitter", dlT.GaussianJitter(**norm_params)),
        # ("Downsample", dl.Downsample(oversample)),
    ]
)

cov = norm_det.Jitter.covariance_matrix
np.linalg.det(cov)

# model = dlT.Toliman(source=src, optics=optics).set("detector", norm_det)

Array(2., dtype=float64)

In [3]:
def covariance_matrix(norm_params):
    """
    Generates the covariance matrix for the multivariate normal distribution.

    Returns
    -------
    covariance_matrix : Array
        The covariance matrix.
    """
    # Compute the rotation angle

    phi = norm_params["phi"]
    shear = norm_params["shear"]
    r = norm_params["r"]

    rot_angle = np.radians(phi)

    # Construct the rotation matrix
    R = np.array(
        [
            [np.cos(rot_angle), -np.sin(rot_angle)],
            [np.sin(rot_angle), np.cos(rot_angle)],
        ]
    )

    # calculating the variances (var1 > var2)
    var1 = np.sqrt(r) / (1 - shear)
    var2 = var1 * (1 - shear) ** 2

    # Construct the skew matrix
    base_matrix = np.array(
        [
            [var1, 0],
            [0, var2],
        ]
    )

    # Compute the covariance matrix
    covariance_matrix = np.dot(np.dot(R, base_matrix), R.T)

    return covariance_matrix

In [4]:
norm_params = {
    "r": 2,
    # "r": 0.25e-4,
    "shear": 0.4,
    "phi": 0.0,
    "kernel_size": 17,
}

cov = covariance_matrix(norm_params)
print(cov)
print(np.linalg.det(cov))
print(1 - (np.diag(cov) ** 0.5)[1] / (np.diag(cov) ** 0.5)[0])

[[2.3570226  0.        ]
 [0.         0.84852814]]
2.0
0.4
